In [1]:
from lima.config import load_config
from lima.dataset import create_dataset_splits, LimaDataset, create_dataloader
from lima.model import load_model, load_tokenizer

/root/learning/papers-from-scratch/LIMA/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

In [2]:
config = load_config()
dataset = create_dataset_splits(config=config)

In [3]:
dataset

{'train': Dataset({
     features: ['conversations', 'source'],
     num_rows: 927
 }),
 'validation': Dataset({
     features: ['conversations', 'source'],
     num_rows: 103
 }),
 'test': Dataset({
     features: ['conversations', 'source'],
     num_rows: 300
 })}

In [5]:
train_dataset = LimaDataset(dataset["train"])
train_dataloader = create_dataloader(train_dataset, batch_size=8, shuffle=True)

In [6]:
tokenizer = load_tokenizer(model_name=config["model"]["name"])

In [12]:
for batch in train_dataloader:
    batch
    break

In [18]:
features = [train_dataset[i] for i in range(4)] 
features

[{'messages': [{'role': 'user',
    'content': "You're given a paragraph from the research paper and your task is to generate a suitable title for the research paper based on the given paper. Under 100 words is a good title length.\n\nThe severe acute respiratory syndrome (SARS) epidemic originating from China in 2002 was caused by a previously uncharacterized coronavirus that could be identified by specific RT-PCR amplification. Efforts to control future SARS outbreaks depend on the accurate and early identification of SARS-CoV infected patients. A real-time fluorogenic RT-PCR assay based on the 3 -noncoding region (3 -NCR) of SARS-CoV genome was developed as a quantitative SARS diagnostic tool. The ideal amplification efficiency of a sensitive SARS-CoV RT-PCR assay should yield an E value (PCR product concentration increase per amplification cycle) equal to 2.0. It was demonstrated that the 3 -NCR SARS-CoV based RT-PCR reactions could be formulated to reach excellent E values of 1.81

In [27]:
tokenizer.apply_chat_template(features[0]["messages"][0]["content"])

[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198]

In [ ]:
def collate_fn(batch):

    # Determine longest sequence in the batch for padding
    sequence_lengths = [len(tokenizer.apply_chat_template(sample["messages"])) for sample in batch]
    max_sequence_length = max(sequence_lengths)

    # Pad sequences

    token_ids = [
        tokenizer.apply_chat_template(sample["messages"]) + [tokenizer.pad_token_id] * (max_sequence_length - len(tokenizer.apply_chat_template(sample["messages"])))
        for sample in batch
    ]

    # Determine response length to infer masking for loss
    length_of_tokenized_responses = [len(tokenizer.apply_chat_template(sample["messages"][-1]["content"])) for sample in batch]

    # Apply mask
    labels = []
    for i, tokenized_sample in enumerate(token_ids):
        mask_first_n_tokens = length_of_tokenized_responses[i]
        labels.append(
            [
                -100 if j < mask_first_n_tokens else token_id
                for j, token_id in enumerate(tokenized_sample)
            ]
        )

    return {
        "token_ids": token_ids,
        "labels": labels,
    }


batch = collate_fn(features)

In [39]:
len(batch["token_ids"])

4